# M10 input-zscore Mondrian calibration — small synthetic benchmark

This notebook evaluates conformal/Mondrian uncertainty calibration for the M10 input-zscore model trained on the 100k/20k/20k/20k split.

M10 uses the same residual dilated architecture as M08, but applies per-sample, per-detector input z-score normalization consistently during training, prediction and real-event inference.

This notebook is a small-scale calibration diagnostic, not yet the final 500k M10 conformal analysis.

## 1. Imports and paths

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import NearestNeighbors

DATA_ROOT = Path("/data/vserrano/cbc_pe_data")
DATASET_ID = "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000"

PRED_PATH = DATA_ROOT / "results" / DATASET_ID / "m10_inputzscore_small_cal_test_predictions_embeddings.npz"

print("PRED_PATH:", PRED_PATH)
print("exists:", PRED_PATH.exists())

PRED_PATH: /data/vserrano/cbc_pe_data/results/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/m10_inputzscore_small_cal_test_predictions_embeddings.npz
exists: True


## 2. Load predictions

In [4]:
data = np.load(PRED_PATH, allow_pickle=True)

print("Files:")
for k in data.files:
    arr = data[k]
    print(f"{k:25s}", arr.shape, arr.dtype)

Files:
y_mean                    (3,) float32
y_std                     (3,) float32
label_names               (3,) <U10
checkpoint_file           () <U196
dataset_path              () <U86
split_path                () <U142
label_stats_path          () <U158
model_config              () object
input_normalization       () object
available_splits          (2,) <U4
pred_cal                  (20000, 3) float32
emb_cal                   (20000, 64) float32
y_cal                     (20000, 3) float32
idx_cal                   (20000,) int64
pred_test                 (20000, 3) float32
emb_test                  (20000, 64) float32
y_test                    (20000, 3) float32
idx_test                  (20000,) int64


## 3. Extract arrays

In [5]:
pred_cal_std = data["pred_cal"].astype(np.float32)
y_cal_std = data["y_cal"].astype(np.float32)
emb_cal = data["emb_cal"].astype(np.float32)

pred_test_std = data["pred_test"].astype(np.float32)
y_test_std = data["y_test"].astype(np.float32)
emb_test = data["emb_test"].astype(np.float32)

y_mean = data["y_mean"].astype(np.float32)
y_std = data["y_std"].astype(np.float32)
label_names = [str(x) for x in data["label_names"].tolist()]

input_normalization = data["input_normalization"].item()

print("label_names:", label_names)
print("input_normalization:", input_normalization)

print("pred_cal_std:", pred_cal_std.shape)
print("y_cal_std:", y_cal_std.shape)
print("emb_cal:", emb_cal.shape)
print("pred_test_std:", pred_test_std.shape)
print("y_test_std:", y_test_std.shape)
print("emb_test:", emb_test.shape)

label_names: ['chirp_mass', 'total_mass', 'chi_eff']
input_normalization: {'enabled': True, 'mode': 'per_sample_per_detector_zscore', 'eps': 1e-06}
pred_cal_std: (20000, 3)
y_cal_std: (20000, 3)
emb_cal: (20000, 64)
pred_test_std: (20000, 3)
y_test_std: (20000, 3)
emb_test: (20000, 64)


## 4. Scale conversion

In [22]:
def inverse_label_standardization(y_std_values, y_mean, y_std):
    return y_std_values * y_std[None, :] + y_mean[None, :]

pred_cal_phys = inverse_label_standardization(pred_cal_std, y_mean, y_std)
y_cal_phys = inverse_label_standardization(y_cal_std, y_mean, y_std)

pred_test_phys = inverse_label_standardization(pred_test_std, y_mean, y_std)
y_test_phys = inverse_label_standardization(y_test_std, y_mean, y_std)

print("pred_test_phys range:")
for j, name in enumerate(label_names):
    print(
        name,
        "true range:", float(y_test_phys[:, j].min()), float(y_test_phys[:, j].max()),
        "pred range:", float(pred_test_phys[:, j].min()), float(pred_test_phys[:, j].max()),
    )

assert input_normalization["enabled"] is True
assert input_normalization["mode"] == "per_sample_per_detector_zscore"

pred_test_phys range:
chirp_mass true range: 4.682109832763672 78.20354461669922 pred range: 5.529243469238281 74.27790069580078
total_mass true range: 10.756988525390625 179.6648406982422 pred range: 12.830734252929688 169.907958984375
chi_eff true range: -0.9976221323013306 0.9986631870269775 pred range: -0.8666300177574158 0.9277456998825073


## 5. Sanities

In [7]:
def regression_metrics(y_true, y_pred, label_names):
    rows = []

    for j, name in enumerate(label_names):
        err = y_pred[:, j] - y_true[:, j]

        mse = np.mean(err**2)
        rmse = np.sqrt(mse)
        mae = np.mean(np.abs(err))
        bias = np.mean(err)

        ss_res = np.sum(err**2)
        ss_tot = np.sum((y_true[:, j] - np.mean(y_true[:, j]))**2)
        r2 = 1.0 - ss_res / ss_tot

        rows.append({
            "label": name,
            "rmse": float(rmse),
            "mae": float(mae),
            "bias": float(bias),
            "r2": float(r2),
        })

    return pd.DataFrame(rows)

metrics_std_df = regression_metrics(
    y_true=y_test_std,
    y_pred=pred_test_std,
    label_names=label_names,
)

metrics_phys_df = regression_metrics(
    y_true=y_test_phys,
    y_pred=pred_test_phys,
    label_names=label_names,
)

display(metrics_std_df)
display(metrics_phys_df)

,label,rmse,mae,bias,r2
0,chirp_mass,0.324498,0.225634,-0.000239,0.897037
1,total_mass,0.294086,0.217628,-0.012974,0.916050
2,chi_eff,0.472071,0.353965,0.039057,0.778215


,label,rmse,mae,bias,r2
0,chirp_mass,5.341313,3.713995,-0.003932,0.897037
1,total_mass,10.157347,7.516605,-0.448099,0.916050
2,chi_eff,0.208007,0.155966,0.017209,0.778215


## 6. Conformal quantile helper

Esta función usa el cuantil finito de split conformal:

q=Quantile
⌈(n+1)(1−α)⌉/n
	​


In [8]:
def conformal_quantile(scores, alpha=0.10):
    """
    Split conformal finite-sample quantile.

    scores: non-negative calibration scores, shape (n,)
    alpha: miscoverage level. alpha=0.10 gives nominal 90% coverage.
    """
    scores = np.asarray(scores, dtype=np.float64)
    n = len(scores)

    if n == 0:
        raise ValueError("Empty calibration scores.")

    q_level = np.ceil((n + 1) * (1.0 - alpha)) / n
    q_level = min(q_level, 1.0)

    return np.quantile(scores, q_level, method="higher")

## 7. Global symmetric conformal baseline

Esto calibra un intervalo global por label en espacio estandarizado, usando: $$s_i=|y_i-\hat{y}_i|$$

In [10]:
ALPHA = 0.10
CL = 1.0 - ALPHA

global_rows = []
global_intervals = {}

for j, name in enumerate(label_names):
    cal_scores = np.abs(y_cal_std[:, j] - pred_cal_std[:, j])
    q = conformal_quantile(cal_scores, alpha=ALPHA)

    lower_test_std = pred_test_std[:, j] - q
    upper_test_std = pred_test_std[:, j] + q

    covered = (y_test_std[:, j] >= lower_test_std) & (y_test_std[:, j] <= upper_test_std)

    width_std = upper_test_std - lower_test_std
    width_phys = width_std * y_std[j]

    global_rows.append({
        "label": name,
        "method": "global_symmetric",
        "q_std": float(q),
        "coverage": float(np.mean(covered)),
        "median_width_std": float(np.median(width_std)),
        "q95_width_std": float(np.quantile(width_std, 0.95)),
        "median_width_phys": float(np.median(width_phys)),
        "q95_width_phys": float(np.quantile(width_phys, 0.95)),
    })

    global_intervals[name] = {
        "lower_test_std": lower_test_std,
        "upper_test_std": upper_test_std,
        "covered": covered,
        "width_std": width_std,
        "width_phys": width_phys,
        "q_std": q,
    }

global_df = pd.DataFrame(global_rows)
display(global_df)

,label,method,q_std,coverage,median_width_std,q95_width_std,median_width_phys,q95_width_phys
0,chirp_mass,global_symmetric,0.528351,0.90435,1.056702,1.056702,17.393578,17.393578
1,total_mass,global_symmetric,0.474761,0.90040,0.949523,0.949523,32.795322,32.795322
2,chi_eff,global_symmetric,0.756548,0.90075,1.513095,1.513095,0.666709,0.666709


## 8. Mondrian por bins de predicción

Primero hacemos una versión sencilla: bins por valor predicho. Para cada label, agrupamos por cuantiles de pred_cal_std[:, j].

In [11]:
def make_quantile_bins(values_cal, values_test, n_bins=12):
    """
    Quantile bins based on calibration values.
    Returns bin ids for cal/test and bin edges.
    """
    values_cal = np.asarray(values_cal)
    values_test = np.asarray(values_test)

    quantiles = np.linspace(0.0, 1.0, n_bins + 1)
    edges = np.quantile(values_cal, quantiles)

    # Avoid duplicate edges.
    edges = np.unique(edges)

    if len(edges) < 3:
        raise ValueError("Too few unique bin edges.")

    # Extend boundaries.
    edges[0] = -np.inf
    edges[-1] = np.inf

    cal_bins = np.digitize(values_cal, edges[1:-1], right=False)
    test_bins = np.digitize(values_test, edges[1:-1], right=False)

    return cal_bins, test_bins, edges

In [12]:
def evaluate_mondrian_symmetric(
    pred_cal_std,
    y_cal_std,
    pred_test_std,
    y_test_std,
    y_std,
    label_names,
    bin_variable_cal,
    bin_variable_test,
    n_bins=12,
    alpha=0.10,
    method_name="mondrian",
):
    rows = []
    interval_store = {}

    for j, name in enumerate(label_names):
        cal_bins, test_bins, edges = make_quantile_bins(
            bin_variable_cal[:, j],
            bin_variable_test[:, j],
            n_bins=n_bins,
        )

        unique_bins = np.unique(cal_bins)

        q_by_bin = {}

        for b in unique_bins:
            mask_cal = cal_bins == b
            scores_b = np.abs(y_cal_std[mask_cal, j] - pred_cal_std[mask_cal, j])
            q_by_bin[int(b)] = conformal_quantile(scores_b, alpha=alpha)

        lower = np.empty(len(pred_test_std), dtype=np.float32)
        upper = np.empty(len(pred_test_std), dtype=np.float32)

        for b in np.unique(test_bins):
            mask_test = test_bins == b

            if int(b) in q_by_bin:
                q_b = q_by_bin[int(b)]
            else:
                # Fallback: global calibration if empty unseen bin.
                q_b = conformal_quantile(
                    np.abs(y_cal_std[:, j] - pred_cal_std[:, j]),
                    alpha=alpha,
                )

            lower[mask_test] = pred_test_std[mask_test, j] - q_b
            upper[mask_test] = pred_test_std[mask_test, j] + q_b

        covered = (y_test_std[:, j] >= lower) & (y_test_std[:, j] <= upper)
        width_std = upper - lower
        width_phys = width_std * y_std[j]

        # Per-bin coverage diagnostics.
        bin_rows = []

        for b in np.unique(test_bins):
            m = test_bins == b
            if m.sum() == 0:
                continue

            bin_rows.append({
                "bin": int(b),
                "n_test": int(m.sum()),
                "coverage": float(np.mean(covered[m])),
                "median_width_std": float(np.median(width_std[m])),
                "median_width_phys": float(np.median(width_phys[m])),
            })

        bin_df = pd.DataFrame(bin_rows)

        rows.append({
            "label": name,
            "method": method_name,
            "n_bins": int(len(unique_bins)),
            "coverage": float(np.mean(covered)),
            "min_coverage_per_bin": float(bin_df["coverage"].min()),
            "max_coverage_per_bin": float(bin_df["coverage"].max()),
            "median_width_std": float(np.median(width_std)),
            "q95_width_std": float(np.quantile(width_std, 0.95)),
            "median_width_phys": float(np.median(width_phys)),
            "q95_width_phys": float(np.quantile(width_phys, 0.95)),
            "n_bins_under_2sigma": int(np.sum(bin_df["coverage"] < CL - 2*np.sqrt(CL*(1-CL)/bin_df["n_test"]))),
        })

        interval_store[name] = {
            "lower_test_std": lower,
            "upper_test_std": upper,
            "covered": covered,
            "width_std": width_std,
            "width_phys": width_phys,
            "test_bins": test_bins,
            "cal_bins": cal_bins,
            "q_by_bin": q_by_bin,
            "bin_df": bin_df,
            "edges": edges,
        }

    return pd.DataFrame(rows), interval_store

In [13]:
mondrian_pred_df, mondrian_pred_store = evaluate_mondrian_symmetric(
    pred_cal_std=pred_cal_std,
    y_cal_std=y_cal_std,
    pred_test_std=pred_test_std,
    y_test_std=y_test_std,
    y_std=y_std,
    label_names=label_names,
    bin_variable_cal=pred_cal_std,
    bin_variable_test=pred_test_std,
    n_bins=12,
    alpha=ALPHA,
    method_name="mondrian_pred_quantile",
)

display(mondrian_pred_df)

,label,method,n_bins,coverage,min_coverage_per_bin,max_coverage_per_bin,median_width_std,q95_width_std,median_width_phys,q95_width_phys,n_bins_under_2sigma
0,chirp_mass,mondrian_pred_quantile,12,0.90465,0.887192,0.921626,1.038300,1.451289,17.090679,23.888582,0
1,total_mass,mondrian_pred_quantile,12,0.90410,0.887421,0.927252,0.853599,1.223192,29.482248,42.247517,0
2,chi_eff,mondrian_pred_quantile,12,0.90330,0.886733,0.922762,1.598252,1.731126,0.704231,0.762779,0


## 9. Embedding difficulty score

Ahora hacemos algo más parecido a tu Mondrian anterior: dificultad local mediante kNN en embedding.

Idea:

1. Para cada punto de calibración, calculamos su residual absoluto.
2. Para cada punto, estimamos dificultad como media de residuales de vecinos en embedding.
3. Creamos bins de dificultad.
4. Calibramos conformal dentro de cada bin.

In [14]:
def compute_knn_difficulty(
    emb_cal,
    residuals_cal_abs,
    emb_query,
    n_neighbors=20,
):
    """
    Difficulty score for each query point based on mean absolute residuals
    of nearest calibration embeddings.

    emb_cal: (N_cal, D)
    residuals_cal_abs: (N_cal,)
    emb_query: (N_query, D)
    """
    nn = NearestNeighbors(
        n_neighbors=n_neighbors,
        algorithm="auto",
        metric="euclidean",
    )
    nn.fit(emb_cal)

    distances, indices = nn.kneighbors(emb_query, return_distance=True)

    difficulty = residuals_cal_abs[indices].mean(axis=1)

    return difficulty

Calculamos dificultad por label

In [21]:
K_NEIGHBORS = 20

difficulty_cal = np.zeros_like(pred_cal_std, dtype=np.float32)
difficulty_test = np.zeros_like(pred_test_std, dtype=np.float32)

for j, name in enumerate(label_names):
    residuals_cal_abs_j = np.abs(y_cal_std[:, j] - pred_cal_std[:, j])

    difficulty_cal[:, j] = compute_knn_difficulty(
        emb_cal=emb_cal,
        residuals_cal_abs=residuals_cal_abs_j,
        emb_query=emb_cal,
        n_neighbors=K_NEIGHBORS,
    )

    difficulty_test[:, j] = compute_knn_difficulty(
        emb_cal=emb_cal,
        residuals_cal_abs=residuals_cal_abs_j,
        emb_query=emb_test,
        n_neighbors=K_NEIGHBORS,
    )

print("difficulty_cal:", difficulty_cal.shape)
print("difficulty_test:", difficulty_test.shape)

for j, name in enumerate(label_names):
    print(name, np.min(difficulty_cal[:, j]), np.median(difficulty_cal[:, j]), np.max(difficulty_cal[:, j]))

difficulty_cal: (20000, 3)
difficulty_test: (20000, 3)
chirp_mass 0.029937664 0.20973784 0.8599553
total_mass 0.059087623 0.20427838 0.65486664
chi_eff 0.10772771 0.32466313 1.5749211


## 10. Exploratory embedding-difficulty Mondrian diagnostic

This section is exploratory. The current difficulty score is estimated from calibration residuals and is not yet split into an independent difficulty-fitting and conformal-calibration subset. Therefore, it should not be used as the selected final conformal system.

In [20]:
mondrian_difficulty_df, mondrian_difficulty_store = evaluate_mondrian_symmetric(
    pred_cal_std=pred_cal_std,
    y_cal_std=y_cal_std,
    pred_test_std=pred_test_std,
    y_test_std=y_test_std,
    y_std=y_std,
    label_names=label_names,
    bin_variable_cal=difficulty_cal,
    bin_variable_test=difficulty_test,
    n_bins=12,
    alpha=ALPHA,
    method_name="mondrian_embedding_difficulty",
)

display(mondrian_difficulty_df)

,label,method,n_bins,coverage,min_coverage_per_bin,max_coverage_per_bin,median_width_std,q95_width_std,median_width_phys,q95_width_phys,n_bins_under_2sigma
0,chirp_mass,mondrian_embedding_difficulty,12,0.83645,0.704064,0.955526,0.741670,2.258286,12.208068,37.171951,8
1,total_mass,mondrian_embedding_difficulty,12,0.83270,0.663513,0.957794,0.712068,1.770128,24.593939,61.138008,8
2,chi_eff,mondrian_embedding_difficulty,12,0.83780,0.600964,0.940843,1.159040,2.934699,0.510703,1.293104,7


## 11. Comparar métodos

In [17]:
summary_df = pd.concat(
    [
        global_df,
        mondrian_pred_df,
        mondrian_difficulty_df,
    ],
    ignore_index=True,
)

display(
    summary_df.sort_values(
        ["label", "method"]
    )
)

,label,method,q_std,coverage,median_width_std,q95_width_std,median_width_phys,q95_width_phys,n_bins,min_coverage_per_bin,max_coverage_per_bin,n_bins_under_2sigma
2,chi_eff,global_symmetric,0.756548,0.90075,1.513095,1.513095,0.666709,0.666709,NaN,NaN,NaN,NaN
8,chi_eff,mondrian_embedding_difficulty,NaN,0.88720,1.365720,2.768002,0.601771,1.219653,12.0,0.832615,0.928527,6.0
5,chi_eff,mondrian_pred_quantile,NaN,0.90330,1.598252,1.731126,0.704231,0.762779,12.0,0.886733,0.922762,0.0
0,chirp_mass,global_symmetric,0.528351,0.90435,1.056702,1.056702,17.393578,17.393578,NaN,NaN,NaN,NaN
6,chirp_mass,mondrian_embedding_difficulty,NaN,0.88580,0.819540,2.022118,13.489836,33.284569,12.0,0.852689,0.926564,7.0
3,chirp_mass,mondrian_pred_quantile,NaN,0.90465,1.038300,1.451289,17.090679,23.888582,12.0,0.887192,0.921626,0.0
1,total_mass,global_symmetric,0.474761,0.90040,0.949523,0.949523,32.795322,32.795322,NaN,NaN,NaN,NaN
7,total_mass,mondrian_embedding_difficulty,NaN,0.88070,0.858880,1.615664,29.664635,55.802994,12.0,0.845150,0.925113,6.0
4,total_mass,mondrian_pred_quantile,NaN,0.90410,0.853599,1.223192,29.482248,42.247517,12.0,0.887421,0.927252,0.0


In [18]:
display(
    summary_df.pivot_table(
        index=["label", "method"],
        values=[
            "coverage",
            "min_coverage_per_bin",
            "median_width_phys",
            "q95_width_phys",
        ],
        aggfunc="first",
    )
)

coverage  median_width_phys  \
label      method                                                       
chi_eff    global_symmetric                0.90075           0.666709   
           mondrian_embedding_difficulty   0.88720           0.601771   
           mondrian_pred_quantile          0.90330           0.704231   
chirp_mass global_symmetric                0.90435          17.393578   
           mondrian_embedding_difficulty   0.88580          13.489836   
           mondrian_pred_quantile          0.90465          17.090679   
total_mass global_symmetric                0.90040          32.795322   
           mondrian_embedding_difficulty   0.88070          29.664635   
           mondrian_pred_quantile          0.90410          29.482248   

                                          min_coverage_per_bin  q95_width_phys  
label      method                                                               
chi_eff    global_symmetric                                NaN        0.666709  
           mondrian_embedding_difficulty              0.832615        1.219653  
           mondrian_pred_quantile                     0.886733        0.762779  
chirp_mass global_symmetric                                NaN       17.393578  
           mondrian_embedding_difficulty              0.852689       33.284569  
           mondrian_pred_quantile                     0.887192       23.888582  
total_mass global_symmetric                                NaN       32.795322  
           mondrian_embedding_difficulty              0.845150       55.802994  
           mondrian_pred_quantile                     0.887421       42.247517

In [23]:
selected_method = "mondrian_pred_quantile"
selected_store = mondrian_pred_store
selected_summary_df = mondrian_pred_df.copy()

display(selected_summary_df)

,label,method,n_bins,coverage,min_coverage_per_bin,max_coverage_per_bin,median_width_std,q95_width_std,median_width_phys,q95_width_phys,n_bins_under_2sigma
0,chirp_mass,mondrian_pred_quantile,12,0.90465,0.887192,0.921626,1.038300,1.451289,17.090679,23.888582,0
1,total_mass,mondrian_pred_quantile,12,0.90410,0.887421,0.927252,0.853599,1.223192,29.482248,42.247517,0
2,chi_eff,mondrian_pred_quantile,12,0.90330,0.886733,0.922762,1.598252,1.731126,0.704231,0.762779,0


## Save results

In [25]:
OUTPUT_DIR = PRED_PATH.parent / "mondrian_m10_inputzscore_small"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

global_df.to_csv(OUTPUT_DIR / "global_symmetric_summary.csv", index=False)
mondrian_pred_df.to_csv(OUTPUT_DIR / "mondrian_pred_quantile_summary.csv", index=False)
mondrian_difficulty_df.to_csv(OUTPUT_DIR / "mondrian_embedding_difficulty_exploratory_summary.csv", index=False)
summary_df.to_csv(OUTPUT_DIR / "mondrian_summary_all_methods.csv", index=False)
selected_summary_df.to_csv(OUTPUT_DIR / "selected_mondrian_pred_quantile_summary.csv", index=False)

print("Saved tables to:", OUTPUT_DIR)

Saved tables to: /data/vserrano/cbc_pe_data/results/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/mondrian_m10_inputzscore_small


In [26]:
selected_intervals_path = OUTPUT_DIR / "selected_mondrian_pred_quantile_intervals_test.npz"

save_dict = {
    "label_names": np.array(label_names),
    "y_mean": y_mean,
    "y_std": y_std,
    "pred_test_std": pred_test_std,
    "y_test_std": y_test_std,
    "pred_test_phys": pred_test_phys,
    "y_test_phys": y_test_phys,
}

for name in label_names:
    store = mondrian_pred_store[name]
    save_dict[f"{name}_lower_test_std"] = store["lower_test_std"]
    save_dict[f"{name}_upper_test_std"] = store["upper_test_std"]
    save_dict[f"{name}_covered"] = store["covered"]
    save_dict[f"{name}_width_std"] = store["width_std"]
    save_dict[f"{name}_width_phys"] = store["width_phys"]
    save_dict[f"{name}_test_bins"] = store["test_bins"]
    save_dict[f"{name}_cal_bins"] = store["cal_bins"]

np.savez_compressed(selected_intervals_path, **save_dict)

print("Saved:", selected_intervals_path)

Saved: /data/vserrano/cbc_pe_data/results/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/mondrian_m10_inputzscore_small/selected_mondrian_pred_quantile_intervals_test.npz


In [27]:
selected_calibration_path = OUTPUT_DIR / "selected_mondrian_pred_quantile_calibration.npz"

calib_save = {
    "label_names": np.array(label_names),
    "y_mean": y_mean,
    "y_std": y_std,
    "alpha": np.array(ALPHA),
    "method": np.array(selected_method),
}

for name in label_names:
    store = mondrian_pred_store[name]

    q_items = sorted(store["q_by_bin"].items())
    q_bins = np.array([b for b, q in q_items], dtype=np.int64)
    q_values = np.array([q for b, q in q_items], dtype=np.float32)

    calib_save[f"{name}_edges"] = store["edges"]
    calib_save[f"{name}_q_bins"] = q_bins
    calib_save[f"{name}_q_values"] = q_values

np.savez_compressed(selected_calibration_path, **calib_save)

print("Saved:", selected_calibration_path)

Saved: /data/vserrano/cbc_pe_data/results/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/mondrian_m10_inputzscore_small/selected_mondrian_pred_quantile_calibration.npz


## Interim selection

For M10-small, the selected conformal system is `mondrian_pred_quantile` with 12 bins and symmetric intervals. It achieves approximately nominal global coverage for all three labels and avoids the undercoverage observed in the exploratory embedding-difficulty system.

This is not yet the final M10 uncertainty model. The next step is to repeat the richer M08-style grid, including asymmetric intervals and stricter difficulty calibration, before training or reporting a final M10-500k conformal result.